# RAG Query Engine and Chatbot Memory

Your current notebook can already retrieve relevant chunks from ChromaDB. The next stage is to turn retrieval into a chatbot pipeline.

The idea is:

1. Keep your hybrid retriever: semantic search plus BM25 keyword search, fused with reciprocal rank fusion.
2. Wrap that retriever in a LlamaIndex `QueryEngine` for single-turn question answering.
3. Wrap the same retriever in a LlamaIndex `ChatEngine` for multi-turn chatbot conversations.
4. Create one memory object per chat, so different chats remember different conversation histories.

This is close to how a production chatbot is organized: retrieval finds evidence, the LLM writes the answer, and chat memory preserves the local conversation context.

In [ ]:
# Install the extra LlamaIndex packages needed for BM25 and Ollama-backed generation.
# Run this once, then restart the kernel if a new import is not recognized.
%pip install -U llama-index-retrievers-bm25 llama-index-llms-ollama ollama

In [2]:
import json
import re
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import chromadb
from chromadb.errors import NotFoundError
from llama_index.core import QueryBundle, Settings, VectorStoreIndex
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.vector_stores.chroma import ChromaVectorStore

resource module not available on Windows


In [3]:
PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
NEWS_DIR = PROJECT_ROOT / "data" / "hk_free_press_news"
CHROMA_DIR = PROJECT_ROOT / "chromadb_store"
SESSION_DIR = PROJECT_ROOT / "session"
COLLECTION_NAME = "news_chat"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
OLLAMA_MODEL = "gemma3:1b"

if not NEWS_DIR.exists():
    raise FileNotFoundError(f"News folder not found: {NEWS_DIR}")

# Store one JSON file per chat session in this folder.
SESSION_DIR.mkdir(parents=True, exist_ok=True)

# LlamaIndex uses Settings as global defaults for retrieval and generation.
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.llm = Ollama(model=OLLAMA_MODEL, request_timeout=120.0)

print(f"News source folder: {NEWS_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Session folder: {SESSION_DIR}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"LLM model: {OLLAMA_MODEL}")

News source folder: C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Session folder: C:\Program Files\Studying\coding\RAG_project\session
Collection name: news_chat
Embedding model: BAAI/bge-small-en-v1.5
LLM model: gemma3:1b


In [4]:
def get_chroma_collection(
    chroma_dir: Path,
    collection_name: str,
    reset_collection: bool = False,
):
    """Create or load a persistent ChromaDB collection."""
    # PersistentClient writes vectors and metadata to disk instead of keeping them in memory.
    chroma_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(chroma_dir))

    # Some ChromaDB versions raise NotFoundError when the collection is missing,
    # while older versions may raise ValueError. Catch both so reset stays safe.
    if reset_collection:
        try:
            client.delete_collection(collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except (NotFoundError, ValueError):
            print(f"Collection did not exist yet: {collection_name}")

    return client.get_or_create_collection(collection_name)


In [5]:
def load_existing_index(
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
) -> VectorStoreIndex:
    """Reconnect to an existing persisted ChromaDB collection without reinserting data."""
    # This is the path you use after closing and reopening the notebook.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection=False)
    vector_store = ChromaVectorStore(chroma_collection=collection)
    return VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)


def load_nodes_from_chroma_collection(collection) -> list[TextNode]:
    """Rebuild TextNode objects from the persisted Chroma collection for keyword retrieval."""
    # Chroma stores text and metadata, so we can reconstruct nodes without rerunning ingestion.
    records = collection.get(include=["documents", "metadatas"])
    documents = records.get("documents", []) or []
    metadatas = records.get("metadatas", []) or []
    ids = records.get("ids", []) or []

    nodes: list[TextNode] = []
    for node_id, document_text, metadata in zip(ids, documents, metadatas):
        if not document_text:
            continue
        nodes.append(
            TextNode(
                id_=node_id,
                text=document_text,
                metadata=metadata or {},
            )
        )

    if not nodes:
        raise ValueError("No stored text nodes were found in the Chroma collection.")

    return nodes


def build_bm25_retriever(collection, top_k: int = 5) -> BM25Retriever:
    """Create a BM25 keyword retriever from the persisted Chroma text and metadata."""
    # BM25 complements embeddings by rewarding exact keyword overlap.
    nodes = load_nodes_from_chroma_collection(collection)
    return BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k)


def reciprocal_rank_fusion(
    semantic_results: list[NodeWithScore],
    keyword_results: list[NodeWithScore],
    top_k: int,
    rrf_k: int = 60,
) -> list[NodeWithScore]:
    """Fuse semantic and keyword rankings using reciprocal rank fusion."""
    # RRF combines rankings instead of raw scores, which makes it robust across retrievers.
    fused_scores: dict[str, float] = defaultdict(float)
    node_lookup: dict[str, NodeWithScore] = {}

    for results in (semantic_results, keyword_results):
        for rank, result in enumerate(results, start=1):
            node_id = result.node.node_id
            fused_scores[node_id] += 1.0 / (rrf_k + rank)

            # Keep the first full node payload so we can print the final fused results later.
            if node_id not in node_lookup:
                node_lookup[node_id] = result

    reranked_results = sorted(
        node_lookup.values(),
        key=lambda result: fused_scores[result.node.node_id],
        reverse=True,
    )

    fused_results: list[NodeWithScore] = []
    for result in reranked_results[:top_k]:
        node_id = result.node.node_id
        fused_results.append(NodeWithScore(node=result.node, score=fused_scores[node_id]))

    return fused_results


def print_ranked_results(results: list[NodeWithScore], title: str) -> None:
    """Print retrieval results with source metadata so ranking quality is easy to inspect."""
    print(f"\n{title}")
    for rank, result in enumerate(results, start=1):
        metadata = result.node.metadata
        file_name = metadata.get("file_name", "unknown file")
        page_label = metadata.get("page_label", metadata.get("page", "unknown page"))
        chunk_number = metadata.get("chunk_number", "unknown chunk")

        print(f"\n--- Result {rank} | score={result.score:.4f} ---")
        print(f"Source: {file_name} | page: {page_label} | chunk: {chunk_number}")
        print(result.node.get_content()[:1200])


def retrieve_relevant_chunks(
    query: str,
    top_k: int = 5,
    query_index: VectorStoreIndex | None = None,
    collection=None,
    bm25_retriever: BM25Retriever | None = None,
    rrf_k: int = 60,
):
    """Run semantic search, keyword search, then fuse both rankings with reciprocal rank fusion."""
    # Load the stored vector index lazily so the function still works after reopening the notebook.
    active_index = query_index if query_index is not None else load_existing_index()

    # Semantic retrieval is strong for meaning and paraphrase matches.
    semantic_retriever = active_index.as_retriever(similarity_top_k=top_k)
    semantic_results = semantic_retriever.retrieve(query)

    # BM25 retrieval is strong for exact terms, names, acronyms, and rare keywords.
    active_collection = collection if collection is not None else get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
    active_bm25_retriever = bm25_retriever if bm25_retriever is not None else build_bm25_retriever(active_collection, top_k=top_k)
    keyword_results = active_bm25_retriever.retrieve(query)

    # Reciprocal rank fusion combines the strengths of both rankings into one final list.
    fused_results = reciprocal_rank_fusion(semantic_results, keyword_results, top_k=top_k, rrf_k=rrf_k)

    # print_ranked_results(semantic_results, "Semantic Search Results")
    # print_ranked_results(keyword_results, "BM25 Keyword Search Results")
    print_ranked_results(fused_results, "Hybrid Search Results (RRF)")

    return {
        "semantic_results": semantic_results,
        "keyword_results": keyword_results,
        "hybrid_results": fused_results,
    }

## Build a LlamaIndex Query Engine, Chat Engine, and Persistent Sessions

A `QueryEngine` is for single-turn RAG: the user asks one question, the retriever finds context, and the LLM writes one answer.

A `ChatEngine` is the next step for a chatbot: it still retrieves context from your vector database, but it also keeps chat memory so follow-up questions can refer to earlier messages.

For persistence, this notebook saves one JSON file per chat in `C:\Program Files\Studying\coding\RAG_project\session`. This is a common product pattern because each chat can be loaded, backed up, deleted, or migrated independently. The JSON stores durable chat data only: metadata plus user and assistant messages. It does not store Python objects, retrievers, embeddings, or the LLM client; those are rebuilt from code when the notebook starts.

In [6]:
class HybridRetriever(BaseRetriever):
    """LlamaIndex retriever that fuses semantic search and BM25 keyword search."""

    def __init__(
        self,
        semantic_retriever,
        keyword_retriever: BM25Retriever,
        final_top_k: int = 5,
        rrf_k: int = 60,
    ):
        # Keep the two retrievers separate so we can inspect or tune them independently.
        self._semantic_retriever = semantic_retriever
        self._keyword_retriever = keyword_retriever
        self._final_top_k = final_top_k
        self._rrf_k = rrf_k
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> list[NodeWithScore]:
        """Retrieve with both methods, then return one fused ranking."""
        # Semantic search finds conceptually similar chunks even when the wording is different.
        semantic_results = self._semantic_retriever.retrieve(query_bundle)

        # BM25 keyword search protects exact terms such as names, dates, acronyms, and places.
        keyword_results = self._keyword_retriever.retrieve(query_bundle)

        # RRF merges ranks without requiring semantic and BM25 scores to be on the same scale.
        return reciprocal_rank_fusion(
            semantic_results,
            keyword_results,
            top_k=self._final_top_k,
            rrf_k=self._rrf_k,
        )


def build_hybrid_retriever(
    query_index: VectorStoreIndex,
    collection,
    final_top_k: int = 5,
    candidate_top_k: int | None = None,
    rrf_k: int = 60,
) -> HybridRetriever:
    """Build the shared retriever used by both the query engine and chat engine."""
    # Pull a slightly larger candidate set before fusion so RRF has room to improve ranking.
    candidate_count = candidate_top_k if candidate_top_k is not None else max(final_top_k * 2, 10)

    semantic_retriever = query_index.as_retriever(similarity_top_k=candidate_count)
    keyword_retriever = build_bm25_retriever(collection, top_k=candidate_count)

    return HybridRetriever(
        semantic_retriever=semantic_retriever,
        keyword_retriever=keyword_retriever,
        final_top_k=final_top_k,
        rrf_k=rrf_k,
    )


def build_rag_query_engine(
    hybrid_retriever: HybridRetriever,
    llm=None,
) -> RetrieverQueryEngine:
    """Create a single-turn RAG query engine from the hybrid retriever."""
    # QueryEngine is best for isolated questions where chat memory is not needed.
    active_llm = llm if llm is not None else Settings.llm
    return RetrieverQueryEngine.from_args(retriever=hybrid_retriever, llm=active_llm)


CHAT_SYSTEM_PROMPT = """
You are a helpful RAG chatbot for an HK Free Press news knowledge base.
Use the retrieved context as your primary evidence.
If the retrieved context does not contain enough information, say that the knowledge base does not have enough evidence.
Use the current chat history to understand follow-up questions, but do not invent facts that are not supported by the retrieved context.
""".strip()


def build_rag_chat_engine(
    hybrid_retriever: HybridRetriever,
    memory: ChatMemoryBuffer,
    llm=None,
) -> ContextChatEngine:
    """Create a multi-turn chat engine that combines retrieval with chat memory."""
    # ChatEngine adds conversation memory around the same retriever used by the query engine.
    active_llm = llm if llm is not None else Settings.llm
    return ContextChatEngine.from_defaults(
        retriever=hybrid_retriever,
        memory=memory,
        llm=active_llm,
        system_prompt=CHAT_SYSTEM_PROMPT,
    )


chat_sessions: dict[str, dict[str, Any]] = {}


def safe_chat_filename(chat_id: str) -> str:
    """Convert a human-readable chat id into a safe JSON file name."""
    # Real products keep user-facing titles separate from storage-safe identifiers.
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", chat_id.strip()).strip("_")
    return safe_name or "default_chat"


def chat_session_path(chat_id: str, session_dir: Path = SESSION_DIR) -> Path:
    """Return the JSON file path used to persist one chat session."""
    # One file per chat keeps sessions easy to inspect, back up, delete, and migrate later.
    return session_dir / f"{safe_chat_filename(chat_id)}.json"


def chat_message_to_dict(message: ChatMessage) -> dict[str, Any]:
    """Convert a LlamaIndex ChatMessage into JSON-serializable data."""
    # Store only durable conversation data, not Python objects or retriever state.
    role = message.role.value if hasattr(message.role, "value") else str(message.role)
    return {
        "role": role,
        "content": message.content or "",
        "additional_kwargs": message.additional_kwargs or {},
    }


def chat_message_from_dict(data: dict[str, Any]) -> ChatMessage:
    """Rebuild a LlamaIndex ChatMessage from saved JSON data."""
    # Convert the saved role string back to LlamaIndex's role enum when possible.
    role_text = data.get("role", "user")
    try:
        role = MessageRole(role_text)
    except ValueError:
        role = role_text

    return ChatMessage(
        role=role,
        content=data.get("content", ""),
        additional_kwargs=data.get("additional_kwargs", {}) or {},
    )


def memory_to_messages(memory: ChatMemoryBuffer) -> list[dict[str, Any]]:
    """Serialize all messages currently stored in one chat memory."""
    # ChatMemoryBuffer owns the current conversation state for one chat session.
    return [chat_message_to_dict(message) for message in memory.get_all()]


def build_memory_from_messages(
    messages: list[dict[str, Any]],
    memory_token_limit: int = 3000,
) -> ChatMemoryBuffer:
    """Create a ChatMemoryBuffer and hydrate it with previously saved messages."""
    # The token limit prevents old conversations from growing beyond the model context window.
    memory = ChatMemoryBuffer.from_defaults(token_limit=memory_token_limit)

    # Reinsert messages in original order so follow-up questions have the same context after reload.
    for message_data in messages:
        memory.put(chat_message_from_dict(message_data))

    return memory


def save_chat_session(chat_id: str, session_dir: Path = SESSION_DIR) -> Path:
    """Persist one active chat session to a JSON file using an atomic write."""
    if chat_id not in chat_sessions:
        raise KeyError(f"Chat session not found: {chat_id}")

    session_dir.mkdir(parents=True, exist_ok=True)
    session_path = chat_session_path(chat_id, session_dir=session_dir)
    temp_path = session_path.with_suffix(".tmp")

    payload = {
        "schema_version": 1,
        "chat_id": chat_id,
        "collection_name": COLLECTION_NAME,
        "embedding_model": EMBED_MODEL_NAME,
        "llm_model": OLLAMA_MODEL,
        "updated_at": datetime.now(timezone.utc).isoformat(),
        "messages": memory_to_messages(chat_sessions[chat_id]["memory"]),
    }

    # Write to a temporary file first, then replace the real file to avoid half-written sessions.
    temp_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    temp_path.replace(session_path)
    return session_path


def load_chat_session(
    chat_id: str,
    hybrid_retriever: HybridRetriever,
    session_dir: Path = SESSION_DIR,
    memory_token_limit: int = 3000,
) -> str:
    """Load one saved chat session and rebuild its LlamaIndex chat engine."""
    session_path = chat_session_path(chat_id, session_dir=session_dir)
    if not session_path.exists():
        raise FileNotFoundError(f"Saved chat session not found: {session_path}")

    payload = json.loads(session_path.read_text(encoding="utf-8"))

    # Saved messages become the memory object; retriever and LLM are rebuilt from current code.
    memory = build_memory_from_messages(
        payload.get("messages", []),
        memory_token_limit=memory_token_limit,
    )
    chat_engine = build_rag_chat_engine(hybrid_retriever=hybrid_retriever, memory=memory)

    chat_sessions[chat_id] = {
        "memory": memory,
        "chat_engine": chat_engine,
        "session_path": session_path,
    }
    return chat_id


def create_chat_session(
    chat_id: str,
    hybrid_retriever: HybridRetriever,
    memory_token_limit: int = 3000,
    load_existing: bool = True,
    overwrite: bool = False,
) -> str:
    """Create a new chat session, or load it from disk when it already exists."""
    session_path = chat_session_path(chat_id)

    # Loading by default matches normal chatbot behavior: reopening a chat restores history.
    if load_existing and session_path.exists() and not overwrite:
        return load_chat_session(
            chat_id=chat_id,
            hybrid_retriever=hybrid_retriever,
            memory_token_limit=memory_token_limit,
        )

    # A new chat starts with empty memory and is immediately saved as an empty session file.
    memory = ChatMemoryBuffer.from_defaults(token_limit=memory_token_limit)
    chat_engine = build_rag_chat_engine(hybrid_retriever=hybrid_retriever, memory=memory)

    chat_sessions[chat_id] = {
        "memory": memory,
        "chat_engine": chat_engine,
        "session_path": session_path,
    }
    save_chat_session(chat_id)
    return chat_id


def chat_with_rag(chat_id: str, message: str):
    """Send a message to one chat session, save the updated history, and print the answer."""
    if chat_id not in chat_sessions:
        raise KeyError(f"Chat session not found: {chat_id}. Create or load it first.")

    # The chat engine uses both retrieved context and this chat's previous messages.
    response = chat_sessions[chat_id]["chat_engine"].chat(message)

    # Save immediately after each assistant response so notebook restarts do not lose turns.
    session_path = save_chat_session(chat_id)

    print(response.response)
    print(f"\nSaved chat session to: {session_path}")
    return response


def show_chat_history(chat_id: str) -> None:
    """Print the stored messages for one chat session."""
    if chat_id not in chat_sessions:
        raise KeyError(f"Chat session not found: {chat_id}")

    # ChatMemoryBuffer stores the user and assistant turns that belong to this chat only.
    messages = chat_sessions[chat_id]["memory"].get_all()
    for message in messages:
        role = message.role.value if hasattr(message.role, "value") else str(message.role)
        print(f"{role}: {message.content}")


def list_saved_chat_sessions(session_dir: Path = SESSION_DIR) -> list[Path]:
    """List saved chat session files on disk."""
    # This is useful after reopening the notebook when you want to see available chats.
    session_dir.mkdir(parents=True, exist_ok=True)
    session_files = sorted(session_dir.glob("*.json"))
    for index, session_file in enumerate(session_files, start=1):
        print(f"{index}. {session_file.name}")
    return session_files


def print_response_sources(response, max_sources: int = 3) -> None:
    """Print source chunks used by a query or chat response."""
    # Source nodes help you debug whether the answer is grounded in the right retrieved text.
    source_nodes = getattr(response, "source_nodes", []) or []
    for rank, source_node in enumerate(source_nodes[:max_sources], start=1):
        metadata = source_node.node.metadata
        article_date = metadata.get("article_date", "unknown date")
        article_title = metadata.get("article_title", metadata.get("file_name", "unknown article"))
        print(f"\nSource {rank}: {article_date} | {article_title}")
        print(source_node.node.get_content()[:800])

## Run the RAG Engines

Run the next cell once per notebook session. It reconnects to ChromaDB, rebuilds the hybrid retriever, and creates a query engine.

Use the query engine for one-off questions. Use the chat engine when the user is in a conversation and may ask follow-up questions like "why?", "summarize that", or "compare it with the first one".

### Single-turn RAG: use the query engine when the question does not need chat history.

In [6]:
# Run this once after reopening the notebook to reconnect to persisted storage.
collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
reloaded_index = load_existing_index()

# Build one shared hybrid retriever and query engine for this notebook session.
hybrid_retriever = build_hybrid_retriever(
    query_index=reloaded_index,
    collection=collection,
    final_top_k=5,
)

In [ ]:
rag_query_engine = build_rag_query_engine(hybrid_retriever)

print(f"Stored vector count: {collection.count()}")
print("Hybrid retriever and RAG query engine are ready.")

sample_query = "Who is Sanae Takaichi ? and what is her relationship with donald trump ?"
query_response = rag_query_engine.query(sample_query)

print(query_response)
print('-' * 70)
print_response_sources(query_response)

Stored vector count: 1499
Hybrid retriever and RAG query engine are ready.
Sanae Takaichi is the Prime Minister of Japan. Her relationship with Donald Trump is that she is a prominent figure in the United States, having met with him during a phone call late on Friday. Trump has warned her to avoid involving South Korea in a military conflict with China, but Takaichi has denied any involvement.
----------------------------------------------------------------------

Source 1: 03-01-2026_Trump invites Japan’s Takaichi to the US early this year.txt, page unknown page
President Donald Trump invited Japan’s Prime Minister Sanae Takaichi to the United States during a phone call late on Friday and they agreed to work towards a meeting early this year, officials said.

Trump has already said he will visit China in April, with Tokyo and Beijing in dispute over Takaichi’s suggestion in November that Japan could intervene militarily in case of any attack on self-ruled Taiwan.

China claims the dem

### Multi-turn RAG With Saved Chat History

The next cells demonstrate a persistent chat. `create_chat_session(...)` loads an existing session file when it already exists, or creates a new JSON file when it does not. Every `chat_with_rag(...)` call saves the updated conversation automatically.

In [6]:
# Run this once after reopening the notebook to reconnect to persisted storage.
collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
reloaded_index = load_existing_index()

# Build one shared hybrid retriever and query engine for this notebook session.
hybrid_retriever = build_hybrid_retriever(
    query_index=reloaded_index,
    collection=collection,
    final_top_k=5,
)

In [7]:
# Create or load one persistent chat session.
# If session/Sanae_Takaichi.json already exists, this restores its previous messages.
chat_id = create_chat_session("Sanae Takaichi", hybrid_retriever)

chat_response = chat_with_rag(
    chat_id=chat_id,
    message="Who is Sanae Takaichi?",
)
print_response_sources(chat_response)

According to the provided context, Sanae Takaichi is the Prime Minister of Japan who won a landslide election in snap elections on Sunday. She is viewed as a conservative and security hawk.

Here’s a breakdown of what the context tells us:

*   **Prime Minister:** He is the leader of Japan.
*   **Recent Political Situation:** He was recently criticized by China, leading to a sharp diplomatic backlash.
*   **Policy:** He’s known for accelerating Japan’s defense strategy, including increased military spending and security cooperation with allies.



Saved chat session to: C:\Program Files\Studying\coding\RAG_project\session\Sanae_Takaichi.json

Source 1: 13-01-2026 | Japan, South Korea leaders meet as China flexes muscles
Japanese Prime Minister Sanae Takaichi will host South Korea’s President Lee Jae Myung for talks on Tuesday aimed at demonstrating their cordial ties as Beijing pressures Tokyo over its stance on Taiwan.

The two leaders will meet in Takaichi’s home region of Nara in we

In [8]:
# Follow-up question: this uses the same saved chat memory, so the pronoun "she" refers to Sanae Takaichi.
follow_up_response = chat_with_rag(
    chat_id=chat_id,
    message="How long did she talk with Donald Trump during a phone call?",
)
print_response_sources(follow_up_response)

The provided context states that Trump and Takaichi “exchanged views mainly on the Indo-Pacific region and confirmed the close cooperation between Japan and the United States” during their phone call.

It doesn’t specify the duration of the call.

Saved chat session to: C:\Program Files\Studying\coding\RAG_project\session\Sanae_Takaichi.json

Source 1: 05-02-2026 | Why did Xi hold back-to-back calls with Putin and Trump
China’s leader Xi Jinping held back-to-back calls with Russia’s Vladimir Putin and US President Donald Trump this week, timing analysts said on Thursday was rare and significant as Beijing positions itself as a stable global power.

Here is what to know about the talks:

Why on the same day?

Xi’s video call with Putin on Wednesday afternoon was followed just hours later by a phone call with Trump.

“The timing of the call is rare and interesting. It is not common for Xi to have two calls with Putin and Trump,” George Chen, a partner at The Asia Group wrote in an online

In [10]:
list_saved_chat_sessions()

1. Sanae_Takaichi.json


[WindowsPath('C:/Program Files/Studying/coding/RAG_project/session/Sanae_Takaichi.json')]

In [11]:
# Inspect memory for the active chat. A different chat_id has a different history file.
show_chat_history(chat_id)

user: Who is Sanae Takaichi?
assistant: According to the provided context, Sanae Takaichi is the Prime Minister of Japan who won a landslide election in snap elections on Sunday. She is viewed as a conservative and security hawk.

Here’s a breakdown of what the context tells us:

*   **Prime Minister:** He is the leader of Japan.
*   **Recent Political Situation:** He was recently criticized by China, leading to a sharp diplomatic backlash.
*   **Policy:** He’s known for accelerating Japan’s defense strategy, including increased military spending and security cooperation with allies.


user: How long did she talk with Donald Trump during a phone call?
assistant: The provided context states that Trump and Takaichi “exchanged views mainly on the Indo-Pacific region and confirmed the close cooperation between Japan and the United States” during their phone call.

It doesn’t specify the duration of the call.


### after creating a session, you can load it after re-opening the notebook

In [7]:
# Run this once after reopening the notebook to reconnect to persisted storage.
collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
reloaded_index = load_existing_index()

# Build one shared hybrid retriever and query engine for this notebook session.
hybrid_retriever = build_hybrid_retriever(
    query_index=reloaded_index,
    collection=collection,
    final_top_k=5,
)

In [8]:
# After reopening the notebook later, rebuild the retriever first, then load the saved chat by id.
# This cell demonstrates the reload path without needing to rerun the previous conversation cells.
loaded_chat_id = load_chat_session("Sanae Takaichi", hybrid_retriever)

In [10]:
follow_up_response = chat_with_rag(
    chat_id=loaded_chat_id,
    message="Have Sanae Takaichi been spoken with Xi Jinping ?",
)
print_response_sources(follow_up_response)

Yes, according to the context, Sanae Takaichi has spoken with Xi Jinping. The text states: “The two leaders, who both took office in 2025, last met in October on the sidelines of the APEC regional summit in Gyeongju in South Korea.”

Saved chat session to: C:\Program Files\Studying\coding\RAG_project\session\Sanae_Takaichi.json

Source 1: 13-01-2026 | Japan, South Korea leaders meet as China flexes muscles
Japanese Prime Minister Sanae Takaichi will host South Korea’s President Lee Jae Myung for talks on Tuesday aimed at demonstrating their cordial ties as Beijing pressures Tokyo over its stance on Taiwan.

The two leaders will meet in Takaichi’s home region of Nara in western Japan, days after Lee visited Chinese leader Xi Jinping in Beijing.

Looming in the background is Japan’s heated diplomatic spat with China, triggered by Takaichi’s suggestion in November that Japan could intervene militarily if China attacks Taiwan.

China, which regards Taiwan as its own territory, reacted angr

In [11]:
show_chat_history(loaded_chat_id)

user: Who is Sanae Takaichi?
assistant: According to the provided context, Sanae Takaichi is the Prime Minister of Japan who won a landslide election in snap elections on Sunday. She is viewed as a conservative and security hawk.

Here’s a breakdown of what the context tells us:

*   **Prime Minister:** He is the leader of Japan.
*   **Recent Political Situation:** He was recently criticized by China, leading to a sharp diplomatic backlash.
*   **Policy:** He’s known for accelerating Japan’s defense strategy, including increased military spending and security cooperation with allies.


user: How long did she talk with Donald Trump during a phone call?
assistant: The provided context states that Trump and Takaichi “exchanged views mainly on the Indo-Pacific region and confirmed the close cooperation between Japan and the United States” during their phone call.

It doesn’t specify the duration of the call.
user: Have Sanae Takaichi been spoken with Xi Jinping ?
assistant: Yes, according 

In [ ]:
# Optional: manually save at any time, even though chat_with_rag saves after every response.
# Optional: manually save at any time, even though chat_with_rag saves after every response.
# Optional: manually save at any time, even though chat_with_rag saves after every response.
saved_path = save_chat_session(chat_id)
print(f"Saved to: {saved_path}")